In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
#Check GPU enabled
import torch
import pandas   as pd
import os
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from pathlib import Path
import shutil
print(torch.cuda.is_available())    

True


In [4]:
# Load the sequences and labels for training
loaded_sequences_df = pd.read_pickle('/content/drive/MyDrive/capstone_team_3/data_set/classic_midi_sequences_256.pkl')
print("MIDI sequences loaded from pickle file.")
print(f"Shape of loaded sequences DataFrame: {loaded_sequences_df.shape}")  

MIDI sequences loaded from pickle file.
Shape of loaded sequences DataFrame: (630987, 2)


In [6]:
train_df, test_df = train_test_split(loaded_sequences_df, test_size=0.2, random_state=42)
print(f"Training set size: {len(train_df)}")
print(f"Testing set size: {len(test_df)}")  

Training set size: 504789
Testing set size: 126198


In [7]:
def cosine_similarity(vec1, vec2):
    vec1 = np.asarray(vec1, dtype=np.float32)
    vec2 = np.asarray(vec2, dtype=np.float32)
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    if norm_vec1 == 0 or norm_vec2 == 0:
        return 0.0
    return float(dot_product / (norm_vec1 * norm_vec2))

def features_to_vector(features):
    features = np.asarray(features, dtype=np.float32)
    if features.ndim == 1:
        features = features.reshape(1, -1)
    summary_vector = np.concatenate([
        features.mean(axis=0),
        features.std(axis=0),
        features.min(axis=0),
        features.max(axis=0)
    ])
    return summary_vector.astype(np.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


### Baseline LSTM sequence classifier
Prepare normalized sequence tensors, train a simple LSTM classifier, and compare learned embeddings with cosine similarity.

In [ ]:
# Prepare normalized tensors and dataloaders
train_sequences = np.stack(train_df['sequence'].to_numpy()).astype(np.float32)
test_sequences = np.stack(test_df['sequence'].to_numpy()).astype(np.float32)

feature_mean = train_sequences.reshape(-1, train_sequences.shape[-1]).mean(axis=0)
feature_std = train_sequences.reshape(-1, train_sequences.shape[-1]).std(axis=0) + 1e-6

# Normalize the sequences
X_train = (train_sequences - feature_mean) / feature_std
X_test = (test_sequences - feature_mean) / feature_std

label_encoder = LabelEncoder()
label_encoder.fit(loaded_sequences_df['label'])
y_train = label_encoder.transform(train_df['label'])
y_test = label_encoder.transform(test_df['label'])

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

batch_size = 64
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=batch_size, shuffle=False)

input_dim = X_train.shape[-1]
sequence_length = X_train.shape[1]
num_classes = len(label_encoder.classes_)

print(f'Train tensor shape: {X_train_tensor.shape}')
print(f'Test tensor shape: {X_test_tensor.shape}')
print(f'Input dimension: {input_dim}, Sequence length: {sequence_length}, Classes: {num_classes}')

Train tensor shape: torch.Size([504789, 256, 3])
Test tensor shape: torch.Size([126198, 256, 3])
Input dimension: 3, Sequence length: 256, Classes: 289


In [9]:
 #Define and train a baseline LSTM classifier
class LSTMSequenceClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, return_embedding=False):
        _, (hidden_state, _) = self.lstm(x)
        embedding = self.dropout(hidden_state[-1])
        logits = self.classifier(embedding)
        if return_embedding:
            return logits, embedding
        return logits



In [ ]:
# Hyperparameters
hidden_dim = 128
learning_rate = 1e-3
epochs = 5



In [12]:
# Initialize model, loss function, and optimizer
model = LSTMSequenceClassifier(input_dim=input_dim, hidden_dim=hidden_dim, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [13]:
history = []
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch_inputs, batch_labels in train_loader:
        batch_inputs = batch_inputs.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()
        logits = model(batch_inputs)
        loss = criterion(logits, batch_labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_inputs.size(0)
        train_correct += (logits.argmax(dim=1) == batch_labels).sum().item()
        train_total += batch_labels.size(0)

    model.eval()
    test_loss = 0.0
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for batch_inputs, batch_labels in test_loader:
            batch_inputs = batch_inputs.to(device)
            batch_labels = batch_labels.to(device)

            logits = model(batch_inputs)
            loss = criterion(logits, batch_labels)

            test_loss += loss.item() * batch_inputs.size(0)
            test_correct += (logits.argmax(dim=1) == batch_labels).sum().item()
            test_total += batch_labels.size(0)

    epoch_metrics = {
        'epoch': epoch + 1,
        'train_loss': train_loss / train_total,
        'train_accuracy': train_correct / train_total,
        'test_loss': test_loss / test_total,
        'test_accuracy': test_correct / test_total
    }
    history.append(epoch_metrics)
    print(
        f"Epoch {epoch + 1}/{epochs} - "
        f"train_loss: {epoch_metrics['train_loss']:.4f}, "
        f"train_accuracy: {epoch_metrics['train_accuracy']:.4f}, "
        f"test_loss: {epoch_metrics['test_loss']:.4f}, "
        f"test_accuracy: {epoch_metrics['test_accuracy']:.4f}"
    )

history_df = pd.DataFrame(history)
history_df

Epoch 1/5 - train_loss: 4.2038, train_accuracy: 0.0952, test_loss: 2.6446, test_accuracy: 0.3344
Epoch 2/5 - train_loss: 1.6353, train_accuracy: 0.5616, test_loss: 0.6104, test_accuracy: 0.8490
Epoch 3/5 - train_loss: 0.4571, train_accuracy: 0.8716, test_loss: 0.0740, test_accuracy: 0.9861
Epoch 4/5 - train_loss: 0.1636, train_accuracy: 0.9551, test_loss: 0.0258, test_accuracy: 0.9951
Epoch 5/5 - train_loss: 0.0839, train_accuracy: 0.9771, test_loss: 0.0107, test_accuracy: 0.9979


,epoch,train_loss,train_accuracy,test_loss,test_accuracy
0,1,4.203760,0.095222,2.644582,0.334403
1,2,1.635299,0.561635,0.610370,0.848991
2,3,0.457108,0.871626,0.073984,0.986141
3,4,0.163644,0.955054,0.025769,0.995119
4,5,0.083850,0.977068,0.010692,0.997868


In [14]:
# Evaluate predictions and compare embeddings with cosine similarity
model.eval()
all_predictions = []
all_targets = []
train_embedding_sums = np.zeros((num_classes, hidden_dim), dtype=np.float32)
train_embedding_counts = np.zeros(num_classes, dtype=np.int32)

with torch.no_grad():
    for batch_inputs, batch_labels in train_loader:
        batch_inputs = batch_inputs.to(device)
        batch_labels = batch_labels.to(device)
        _, embeddings = model(batch_inputs, return_embedding=True)
        embeddings = embeddings.cpu().numpy()
        labels_np = batch_labels.cpu().numpy()

        for embedding, label in zip(embeddings, labels_np):
            train_embedding_sums[label] += embedding
            train_embedding_counts[label] += 1

    for batch_inputs, batch_labels in test_loader:
        batch_inputs = batch_inputs.to(device)
        batch_labels = batch_labels.to(device)
        logits, _ = model(batch_inputs, return_embedding=True)
        predictions = logits.argmax(dim=1)
        all_predictions.extend(predictions.cpu().numpy())
        all_targets.extend(batch_labels.cpu().numpy())

prototype_embeddings = train_embedding_sums / np.maximum(train_embedding_counts[:, None], 1)
test_accuracy = accuracy_score(all_targets, all_predictions)
print(f'Test accuracy: {test_accuracy:.4f}')

prediction_preview = pd.DataFrame({
    'true_label': label_encoder.inverse_transform(all_targets[:10]),
    'predicted_label': label_encoder.inverse_transform(all_predictions[:10])
})
prediction_preview



Test accuracy: 0.9979


,true_label,predicted_label
0,burg_spinnerlied.mid,burg_spinnerlied.mid
1,bor_ps7.mid,bor_ps7.mid
2,liz_et_trans4.mid,liz_et_trans4.mid
3,appass_1.mid,appass_1.mid
4,liz_et_trans4.mid,liz_et_trans4.mid
5,liz_et6.mid,liz_et6.mid
6,mz_570_1.mid,mz_570_1.mid
7,schub_d960_1.mid,schub_d960_1.mid
8,brahms_opus1_4.mid,brahms_opus1_4.mid
9,schumm-2.mid,schumm-2.mid


In [15]:
sample_index = 0
sample_sequence = X_test_tensor[sample_index:sample_index + 1].to(device)
sample_true_label = label_encoder.inverse_transform([y_test[sample_index]])[0]
import sklearn.metrics.pairwise  as pairwise
with torch.no_grad():
    sample_logits, sample_embedding = model(sample_sequence, return_embedding=True)
    sample_prediction = int(sample_logits.argmax(dim=1).item())
    sample_embedding = sample_embedding.squeeze(0).cpu().numpy()
# Sample embedding 
print(f'Sample true label: {sample_true_label}')
print(f"Sample predicted label: {label_encoder.inverse_transform([sample_prediction])[0]}")
print( "sample embedding shape:", sample_embedding.shape)
similarity_rows = []
for class_index, class_name in enumerate(label_encoder.classes_):
    similarity_rows.append({
        'label': class_name,
        'cosine_similarity': pairwise.cosine_similarity(sample_embedding.reshape(1, -1), prototype_embeddings[class_index].reshape(1, -1))[0][0]
    })



Sample true label: burg_spinnerlied.mid
Sample predicted label: burg_spinnerlied.mid
sample embedding shape: (128,)


In [16]:
similarity_df = pd.DataFrame(similarity_rows).sort_values('cosine_similarity', ascending=False).head(5)
print(f'Sample true label: {sample_true_label}')
print(f"Sample predicted label: {label_encoder.inverse_transform([sample_prediction])[0]}")
similarity_df

Sample true label: burg_spinnerlied.mid
Sample predicted label: burg_spinnerlied.mid


,label,cosine_similarity
58,burg_spinnerlied.mid,0.783696
89,chpn_op23.mid,0.545800
243,schuim-2.mid,0.525158
270,scn16_7.mid,0.513604
205,mz_331_1.mid,0.475677


In [18]:
# Load the sequences and labels for testing on external MIDI files
external_test_df = pd.read_pickle('/content/drive/MyDrive/capstone_team_3/data_set/external_test_sequences.pkl')
print("External MIDI sequences loaded from pickle file.")
print(f"Shape of external test sequences DataFrame: {external_test_df.shape}")
print("Sample external test sequences:")
external_test_df.head()

External MIDI sequences loaded from pickle file.
Shape of external test sequences DataFrame: (203723, 2)
Sample external test sequences:


,sequence,label
0,"[[0.7999999999999998, 0.0, 44.0], [0.300000000...",Piano Sonata n12 K332.mid
1,"[[0.30000000000000027, 4.0, 47.0], [0.79999999...",Piano Sonata n12 K332.mid
2,"[[0.7999999999999998, 3.0, 49.0], [0.304166666...",Piano Sonata n12 K332.mid
3,"[[0.3041666666666667, -3.0, 52.0], [0.79999999...",Piano Sonata n12 K332.mid
4,"[[0.7999999999999998, 1.0, 54.0], [0.299999999...",Piano Sonata n12 K332.mid


In [ ]:
# Convert merged_test_sequences to X_test_external and run inference (preserve X_test for Music-BERT)
X_test_external_sequences = np.stack(external_test_df['sequence'].to_numpy()).astype(np.float32)
X_test_external = (X_test_external_sequences - feature_mean) / feature_std

In [ ]:


# Predict label for each test sequence using cosine similarity to class prototypes
model.eval()
predicted_labels = []
with torch.no_grad():
    for seq in X_test_external:
        seq_tensor = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)
        _, seq_embedding = model(seq_tensor, return_embedding=True)
        seq_embedding = seq_embedding.squeeze(0).cpu().numpy()

        similarities = [
            pairwise.cosine_similarity(seq_embedding.reshape(1, -1), prototype_embeddings[i].reshape(1, -1))[0][0]
            for i in range(num_classes)
        ]
        predicted_label_index = int(np.argmax(similarities))
        predicted_labels.append(label_encoder.inverse_transform([predicted_label_index])[0])


: 

In [ ]:
external_test_df['predicted_label'] = predicted_labels
print("Sample test sequences with predicted labels:")
print(external_test_df.head())

In [ ]:
# Stoe the  results in a pickle file for later analysis (only labels and predicted labels)
external_test_df[['label', 'predicted_label']].to_pickle('/content/drive/MyDrive/capstone_team_3/data_set/external_test_predictions.pkl')
print("External test predictions saved to pickle file.")

In [ ]:
# Save the model state for later use
model_save_path = '/content/drive/MyDrive/capstone_team_3/models/lstm_sequence_classifier.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Model state saved to {model_save_path}")